# Assignment 2 – Train a Machine Learning Model Using Amazon SageMaker

**Student:** Izzet Abidi (300898230)
**Course:** COMP 264 – Cloud ML | Centennial College
**Due:** Friday, Week 12

## Objective

Train an XGBoost binary classifier on the UCI Adult Census Income dataset using Amazon SageMaker.
The model predicts whether an individual earns more than $50K per year based on demographic features.

### Steps Covered
1. Install and configure the SageMaker SDK
2. Download, explore, and transform the dataset
3. Train a model with custom hyperparameters
4. Deploy the model to a real-time endpoint
5. Evaluate model performance (confusion matrix, ROC AUC, R²)
6. Clean up all AWS resources

## Step 1 – Install / Update SageMaker SDK

In [ ]:
!pip install -qU sagemaker

## Step 2 – Imports

In [ ]:
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sagemaker

from sagemaker.serializers import CSVSerializer
from sagemaker.session import TrainingInput
from sklearn.model_selection import train_test_split
from sklearn import metrics

print("SageMaker SDK version:", sagemaker.__version__)

## Step 3 – Define the S3 Prefix

The prefix identifies where training data and model artifacts are stored in the default SageMaker bucket.

In [ ]:
prefix = "demo-sage-maker-xgboost-izzet"
print("S3 Prefix:", prefix)

## Step 4 – Download, Explore, and Transform the Dataset

Load the UCI Adult Census Income dataset. The dataset has two files:
- `adult.data` – training portion
- `adult.test` – test portion (has a trailing period on labels that must be stripped)

Rows with unknown values (`?`) are dropped. Categorical features are one-hot encoded.

In [ ]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income",
]

train_df = pd.read_csv(
    "adult.data", names=columns, header=None,
    skipinitialspace=True, na_values="?",
)
test_df = pd.read_csv(
    "adult.test", names=columns, header=None,
    skiprows=1, skipinitialspace=True, na_values="?",
)

# strip whitespace from text columns
for frame in (train_df, test_df):
    for col in frame.select_dtypes(include="object").columns:
        frame[col] = frame[col].str.strip()

# remove trailing period from test labels
test_df["income"] = test_df["income"].str.rstrip(".")

# combine and drop unknowns
df = pd.concat([train_df, test_df], ignore_index=True)
rows_before = len(df)
df = df.dropna().reset_index(drop=True)

print(f"Loaded {rows_before} rows from raw files.")
print(f"Dropped {rows_before - len(df)} rows with unknown values.")
print(f"Working with {len(df)} clean rows.")

### Create the Target Variable and Print Required Information

In [ ]:
# create binary target: 1 if income >50K, 0 otherwise
df["income_over_50k"] = (df["income"] == ">50K").astype(int)

print("Prefix name:", prefix)
print("Target variable type:", type(df["income_over_50k"]))
print("Target variable dtype:", df["income_over_50k"].dtype)
print("Target variable size:", len(df["income_over_50k"]))
print("\nClass value counts:")
print(df["income_over_50k"].value_counts())

### One-Hot Encode Categorical Features

The features used for the model exclude `fnlwgt` and `education` (keeping `education_num` instead). Categorical columns are converted to binary indicator columns.

In [ ]:
model_features = [
    "age", "workclass", "education_num", "marital_status", "occupation",
    "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country",
]
categorical_features = [
    "workclass", "marital_status", "occupation", "relationship",
    "race", "sex", "native_country",
]

features = pd.get_dummies(df[model_features], columns=categorical_features, dtype=int)
labels = df["income_over_50k"]

print("Encoded feature matrix shape:", features.shape)
print("Positive class rate:", f"{labels.mean():.4f}")

### Explore Numeric Features

In [ ]:
numeric_features = ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
df[numeric_features].hist(bins=30, sharey=True, figsize=(16, 8))
plt.tight_layout()
plt.show()

### Train / Validation / Test Split

Split into 60% train, 20% validation, 20% test. Stratified on the target to preserve class balance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=1, stratify=labels,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=1, stratify=y_train,
)

print(f"Train rows:      {len(X_train)}")
print(f"Validation rows: {len(X_val)}")
print(f"Test rows:       {len(X_test)}")

### Write SageMaker-Format CSV Files

SageMaker's built-in XGBoost expects the target label in the first column, followed by feature values, with no header row.

In [ ]:
def label_first(labels_series, features_df):
    """Place the target label in the first column (SageMaker XGBoost format)."""
    return pd.concat([
        labels_series.reset_index(drop=True).rename("income_over_50k").astype(int),
        features_df.reset_index(drop=True),
    ], axis=1)

train_data = label_first(y_train, X_train)
val_data = label_first(y_val, X_val)
test_data = label_first(y_test, X_test)

train_data.to_csv("train.csv", index=False, header=False)
val_data.to_csv("validation.csv", index=False, header=False)
X_test.to_csv("test.csv", index=False, header=False)

print("Wrote train.csv:", train_data.shape)
print("Wrote validation.csv:", val_data.shape)
print("Wrote test.csv:", X_test.shape)

## Step 5 – Start the SageMaker Session

Discover the AWS region, execution role, and default S3 bucket from the notebook environment.

In [ ]:
session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()
bucket = session.default_bucket()

print("AWS Region:", region)
print("Role ARN:", role)
print("Default S3 bucket:", bucket)

### Upload CSV Files to S3

In [ ]:
s3 = boto3.Session().resource("s3")

for file_name in ["train.csv", "validation.csv", "test.csv"]:
    s3_key = f"{prefix}/data/{file_name}"
    s3.Bucket(bucket).Object(s3_key).upload_file(file_name)
    print(f"Uploaded {file_name} -> s3://{bucket}/{s3_key}")

train_s3_uri = f"s3://{bucket}/{prefix}/data/train.csv"
val_s3_uri = f"s3://{bucket}/{prefix}/data/validation.csv"
test_s3_uri = f"s3://{bucket}/{prefix}/data/test.csv"

## Step 6 – Train the Model

Configure the SageMaker built-in XGBoost algorithm with the assignment-specified hyperparameters and launch the training job.

**Hyperparameters (as specified in assignment):**
- `max_depth = 4`
- `eta = 0.25`
- `gamma = 4`
- `min_child_weight = 4`
- `subsample = 0.7`
- `objective = "binary:logistic"`
- `num_round = 500`

In [ ]:
# retrieve the XGBoost container image URI for the current region
image_uri = sagemaker.image_uris.retrieve("xgboost", region, "1.2-1")
output_path = f"s3://{bucket}/{prefix}/xgboost-model"

xgb_estimator = sagemaker.estimator.Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    volume_size=5,
    output_path=output_path,
    sagemaker_session=session,
)

xgb_estimator.set_hyperparameters(
    max_depth=4,
    eta=0.25,
    gamma=4,
    min_child_weight=4,
    subsample=0.7,
    objective="binary:logistic",
    num_round=500,
)

In [ ]:
# launch the training job
train_input = TrainingInput(train_s3_uri, content_type="csv")
validation_input = TrainingInput(val_s3_uri, content_type="csv")

xgb_estimator.fit({"train": train_input, "validation": validation_input}, wait=True)

print("Training job name:", xgb_estimator.latest_training_job.job_name)
print("Model artifact:", xgb_estimator.model_data)

## Step 7 – Deploy the Model

Deploy the trained model to a real-time SageMaker endpoint. **The endpoint is billable while running** – delete it during cleanup.

In [ ]:
xgb_predictor = xgb_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium",
    serializer=CSVSerializer(),
)
print("Endpoint name:", xgb_predictor.endpoint_name)

## Step 8 – Evaluate the Model

Send the test set to the endpoint in batches, collect prediction probabilities, and compute classification metrics.

In [ ]:
def predict_batches(predictor, feature_frame, batch_size=1000):
    """Send predictions to endpoint in manageable batches."""
    split_arrays = np.array_split(
        feature_frame.to_numpy(),
        int(feature_frame.shape[0] / float(batch_size) + 1),
    )
    predictions = ""
    for array in split_arrays:
        if len(array) == 0:
            continue
        response = predictor.predict(array).decode("utf-8")
        predictions = ",".join([predictions, response])
    return np.fromstring(predictions[1:], sep=",")

predictions = predict_batches(xgb_predictor, X_test)
print(f"Received {len(predictions)} predictions")

In [ ]:
# classification metrics at cutoff 0.5
cutoff = 0.5
hard_preds = np.where(predictions > cutoff, 1, 0)

print("Confusion Matrix:")
print(metrics.confusion_matrix(y_test, hard_preds))
print("\nClassification Report:")
print(metrics.classification_report(y_test, hard_preds))

accuracy = metrics.accuracy_score(y_test, hard_preds)
roc_auc = metrics.roc_auc_score(y_test, predictions)
print(f"Accuracy at cutoff {cutoff}: {accuracy:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")

In [ ]:
# prediction probability histogram
plt.figure(figsize=(10, 6))
plt.hist(predictions, bins=30)
plt.title("Predicted Probabilities Distribution")
plt.xlabel("Prediction")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

### Calculate R-Squared

R² measures how much variance in the target is explained by the model's predictions. While R² is typically used for regression, the assignment requires it here for the binary classification output.

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, predictions)
print(f"R-squared: {r2:.4f}")

### Log Loss by Cutoff

In [ ]:
cutoffs = np.arange(0.01, 1.0, 0.01)
log_losses = [
    metrics.log_loss(y_test, np.where(predictions > c, 1, 0))
    for c in cutoffs
]

plt.figure(figsize=(12, 6))
plt.plot(cutoffs, log_losses)
plt.title("Log Loss by Cutoff")
plt.xlabel("Cutoff")
plt.ylabel("Log Loss")
plt.tight_layout()
plt.show()

best_idx = int(np.argmin(log_losses))
print(f"Best cutoff by log loss: {cutoffs[best_idx]:.2f}")
print(f"Minimum log loss: {log_losses[best_idx]:.4f}")

## Submission Summary

Fill in after execution:

- **AWS Region:**
- **Notebook Instance Name:** izzetsession
- **S3 Bucket:**
- **S3 Prefix:** demo-sage-maker-xgboost-izzet
- **Training Job Name:**
- **Endpoint Name:**
- **Accuracy at cutoff 0.5:**
- **ROC AUC:**
- **R-squared:**
- **Best Cutoff:**

## Step 9 – Cleanup

**Execute these steps immediately after collecting all evidence and recording the demo video.**

1. Delete the endpoint (stops billing)
2. Delete items in S3 bucket (data and model)
3. Stop the notebook instance
4. Delete the notebook instance

In [ ]:
# delete the endpoint
xgb_predictor.delete_endpoint()
print("Endpoint deleted.")

In [ ]:
# delete S3 objects under the demo prefix
s3_bucket = boto3.resource("s3").Bucket(bucket)
objects = list(s3_bucket.objects.filter(Prefix=prefix))
if objects:
    s3_bucket.delete_objects(Delete={"Objects": [{"Key": obj.key} for obj in objects]})
    print(f"Deleted {len(objects)} objects from s3://{bucket}/{prefix}/")
else:
    print("No objects found to delete.")

**After running the cleanup cells above:**
1. Navigate to SageMaker in the AWS Console
2. Go to **Notebook instances** and **Stop** the `izzetsession` instance
3. Once stopped, **Delete** the notebook instance
4. Record your demo video now (explain the notebook from your local machine, then show the SageMaker training job history and S3 bucket)